# TRIBE v2 — Colab Backend
**Run Cell 1 → runtime restarts automatically → then run Cells 2–7**

In [ ]:
# CELL 1 — Install everything + restart runtime (restart is required to flush old numpy from memory)
!pip install -q uv
!uv pip install --system "tribev2[plotting] @ git+https://github.com/facebookresearch/tribev2.git"
!uv pip install --system fastapi "uvicorn[standard]" yt-dlp pydantic nest_asyncio huggingface_hub
!apt-get install -y -q ffmpeg 2>/dev/null || true

print("\n✓ Install complete. Restarting runtime to flush old numpy from memory...")
import os, time; time.sleep(2)
os.kill(os.getpid(), 9)  # Colab auto-restarts after this — expected behaviour

In [ ]:
# CELL 2 — Verify imports (run after runtime restarts)
from tribev2.demo_utils import TribeModel
import tribev2, numpy as np, torch
print(f"tribev2 : {tribev2.__file__}")
print(f"numpy   : {np.__version__}")
print(f"torch   : {torch.__version__}")
print("TribeModel ✓ — all imports OK")

In [ ]:
# CELL 3 — Clone / update brain-neuro
import os
repo = '/content/brain-neuro'
if os.path.exists(repo):
    !git -C {repo} pull -q
    print('brain-neuro updated ✓')
else:
    !git clone -q https://github.com/Sambhram1/brain-neuro.git {repo}
    print('brain-neuro cloned ✓')
os.chdir(repo)
print(f'cwd: {os.getcwd()}')

In [ ]:
# CELL 4 — HuggingFace login (weights download automatically on first request)
import huggingface_hub
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    print('HF_TOKEN from Colab secrets ✓')
except Exception:
    hf_token = input('Enter HuggingFace token (huggingface.co/settings/tokens): ')
huggingface_hub.login(token=hf_token, add_to_git_credential=False)
print('Logged in ✓')

In [ ]:
# CELL 5 (Optional) — cookies.txt for Instagram/TikTok. Skip for YouTube.
import shutil, os
try:
    from google.colab import files
    print('Upload cookies.txt or skip this cell')
    uploaded = files.upload()
    for fname in uploaded:
        shutil.move(fname, 'cookies.txt')
        print('cookies.txt saved ✓')
except ImportError:
    print('cookies.txt found ✓' if os.path.exists('cookies.txt') else 'No cookies.txt — YouTube only')

In [ ]:
# CELL 6 — Install cloudflared tunnel
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 \
    -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
print('cloudflared ✓')

In [ ]:
# CELL 7 — Start FastAPI backend + public tunnel
import nest_asyncio, uvicorn, threading, subprocess, time, re, os
os.chdir('/content/brain-neuro')
nest_asyncio.apply()

def run_server():
    uvicorn.run('backend:app', host='0.0.0.0', port=8000, log_level='warning')

threading.Thread(target=run_server, daemon=True).start()
time.sleep(4)
print('FastAPI running on :8000 ✓')

proc = subprocess.Popen(
    ['/usr/local/bin/cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
print('Waiting for tunnel...')
for line in proc.stdout:
    match = re.search(r'https://[\w-]+\.trycloudflare\.com', line.decode())
    if match:
        url = match.group()
        print(f'\n{"═"*52}')
        print(f'  BACKEND URL: {url}')
        print(f'{"═"*52}')
        print('→ Paste into the frontend Backend field')
        break